In [1]:
import pandas as pd
import random
from collections import Counter

# Cargar el corpus
df = pd.read_csv("../data/processed/reports_cleaned.csv")

# Tomar el texto ORIGINAL (no el limpio normalizado), 
# porque las regex van a actuar sobre informes reales
TEXTO_COL = "Full_Report"  # columna original, no Full_Report_clean
LABEL_COL = "BI-RADS"

# Si tu CSV no tiene Full_Report, usa la limpia:
if TEXTO_COL not in df.columns:
    TEXTO_COL = "Full_Report_clean"
    print(f"Usando columna {TEXTO_COL} (texto normalizado)")

print(f"Total informes: {len(df)}")
print(f"Distribución:\n{df[LABEL_COL].value_counts().sort_index()}")
print(f"\nColumnas disponibles: {df.columns.tolist()}")

Total informes: 4357
Distribución:
BI-RADS
0     966
1     596
2    2635
3      87
4      52
5      16
6       5
Name: count, dtype: int64

Columnas disponibles: ['Full_Report', 'Conclusion', 'Recommendations', 'BI-RADS', 'Full_Report_clean', 'Conclusion_clean', 'Recommendations_clean', 'len_report']


In [8]:
# Muestreo balanceado: 2 ejemplos de cada BI-RADS
random.seed(42)

print("=" * 80)
print("MUESTRA DE INFORMES REALES POR CATEGORÍA BI-RADS")
print("=" * 80)

for birads in sorted(df[LABEL_COL].unique()):
    grupo = df[df[LABEL_COL] == birads]
    n_ejemplos = min(2, len(grupo))
    muestra = grupo.sample(n=n_ejemplos, random_state=42)
    
    print(f"\n{'#' * 80}")
    print(f"# BI-RADS {birads}  (total en corpus: {len(grupo)})")
    print(f"{'#' * 80}")
    
    for i, (idx, row) in enumerate(muestra.iterrows(), 1):
        print(f"\n--- Ejemplo {i} (índice {idx}) ---")
        print(f"\n[Full_Report]")
        print(row["Full_Report"])
        print(f"\n[Conclusion]")
        print(row["Conclusion"])
        print(f"\n[Recommendations]")
        print(row["Recommendations"])
        print()

MUESTRA DE INFORMES REALES POR CATEGORÍA BI-RADS

################################################################################
# BI-RADS 0  (total en corpus: 966)
################################################################################

--- Ejemplo 1 (índice 1146) ---

[Full_Report]
MAMOGRAFÍA DIGITALIZADA BILATERAL CRÁNEO-CAUDAL Y MEDIO LATERAL OBLÍCUAS
NO SE OBSERVAN ALTERACIONES CUTÁNEAS NI DEL MAMELÓN.
LAS MAMAS SE PRESENTAN HETEROGÉNEAMENTE DENSAS QUE PODRÍA OCULTAR ALGUNOS NÓDULOS PEQUEÑOS (MAMAS TIPO C).
EN AMBAS MAMAS IMPRESIONAN ALGUNAS OPACIDADES NODULARES OVALADAS, ISODENSAS Y DE MÁRGENES OSCURECIDOS, QUE MIDEN ENTRE 3 Y 8 MM DE DIÁMETRO MAYOR.
NO SE OBSERVAN CALCIFICACIONES SOSPECHOSAS.
SE VISUALIZAN CALCIFICACIONES PUNTIFORMES AISLADAS EN AMBAS MAMAS.
IMÁGENES NODULARES CON CARACTERES GANGLIONARES EN AMBAS REGIONES PECTOAXILARES.
CONCLUSIÓN:
- MAMAS CON ÁREAS DENSAS.
- OPACIDADES NODULARES DE NATURALEZA A DETERMINAR EN AMBAS MAMAS.
- CALCIFICACIONES CON CARACTE

Debido a la incongruencia del mismo paciente del campo recomendacion y full reporte, se hacen pruebas.

In [10]:
# Versión defensiva — no falla aunque el caso problemático no exista

print("=" * 70)
print("DIAGNÓSTICO DE LA COLUMNA Recommendations")
print("=" * 70)

# 1) Conteo de valores 'sin_recomendacion'
n_sin_rec = (df["Recommendations"] == "sin_recomendacion").sum()
n_con_rec = len(df) - n_sin_rec
print(f"\nTotal informes: {len(df)}")
print(f"  Con valor 'sin_recomendacion': {n_sin_rec} ({100*n_sin_rec/len(df):.1f}%)")
print(f"  Con recomendación poblada:     {n_con_rec} ({100*n_con_rec/len(df):.1f}%)")

# 2) Full_Report con la palabra 'recomendaci'
df["tiene_recom_en_full"] = df["Full_Report"].str.contains(
    r"recomendaci", case=False, regex=True, na=False
)
n_full_con_recom = df["tiene_recom_en_full"].sum()
print(f"\nFull_Report contiene 'recomendaci...': {n_full_con_recom} ({100*n_full_con_recom/len(df):.1f}%)")

# 3) Cruce: caso problemático
mask_problematico = (
    (df["Recommendations"] == "sin_recomendacion") &
    (df["tiene_recom_en_full"])
)
n_problematico = mask_problematico.sum()
print(f"\n>>> CASO PROBLEMÁTICO <<<")
print(f"Recommendations='sin_recomendacion' PERO Full_Report sí menciona recomendaciones:")
print(f"   {n_problematico} informes ({100*n_problematico/len(df):.1f}%)")

# 4) Caso simétrico también vale la pena
mask_simetrico = (
    (df["Recommendations"] != "sin_recomendacion") &
    (~df["tiene_recom_en_full"])
)
n_simetrico = mask_simetrico.sum()
print(f"\n>>> CASO SIMÉTRICO (curiosidad) <<<")
print(f"Recommendations poblado PERO Full_Report no menciona 'recomendaciones':")
print(f"   {n_simetrico} informes ({100*n_simetrico/len(df):.1f}%)")

# 5) Mostrar ejemplos solo si existen
print("\n" + "=" * 70)
print("EJEMPLOS")
print("=" * 70)

if n_problematico > 0:
    print(f"\n--- Caso problemático (mostrar hasta 2) ---")
    for idx, row in df[mask_problematico].sample(min(2, n_problematico), random_state=42).iterrows():
        print(f"\nÍndice {idx} | BI-RADS {row['BI-RADS']}")
        print(f"Recommendations: {repr(row['Recommendations'])}")
        print(f"Full_Report (últimos 250 chars):")
        print(f"   ...{row['Full_Report'][-250:]}")
else:
    print("\n>>> No hay caso problemático. La limpieza del nb01 funcionó correctamente. <<<")

# 6) Como complemento: mostrar 2 ejemplos donde Recommendations SÍ tiene contenido real
print(f"\n\n--- Ejemplos donde Recommendations está poblado correctamente ---")
con_recom = df[df["Recommendations"] != "sin_recomendacion"]
if len(con_recom) > 0:
    for idx, row in con_recom.sample(min(2, len(con_recom)), random_state=42).iterrows():
        print(f"\nÍndice {idx} | BI-RADS {row['BI-RADS']}")
        print(f"Recommendations: {row['Recommendations'][:300]}")
        print(f"---")
        print(f"Full_Report (últimos 250 chars):")
        print(f"   ...{row['Full_Report'][-250:]}")
else:
    print("Ningún informe tiene Recommendations poblado distinto de 'sin_recomendacion'.")

DIAGNÓSTICO DE LA COLUMNA Recommendations

Total informes: 4357
  Con valor 'sin_recomendacion': 10 (0.2%)
  Con recomendación poblada:     4347 (99.8%)

Full_Report contiene 'recomendaci...': 4347 (99.8%)

>>> CASO PROBLEMÁTICO <<<
Recommendations='sin_recomendacion' PERO Full_Report sí menciona recomendaciones:
   0 informes (0.0%)

>>> CASO SIMÉTRICO (curiosidad) <<<
Recommendations poblado PERO Full_Report no menciona 'recomendaciones':
   0 informes (0.0%)

EJEMPLOS

>>> No hay caso problemático. La limpieza del nb01 funcionó correctamente. <<<


--- Ejemplos donde Recommendations está poblado correctamente ---

Índice 2975 | BI-RADS 0
Recommendations: - Se sugiere ecografía mamaria para posterior recategorización.
---
Full_Report (últimos 250 chars):
   ...ALORACIÓN:
- Mamas con áreas densas.
- Opacidad nodular de naturaleza a determinar en la mama derecha.
- BI-RADS ® 0 (según la ACR). Requerirá de estudio complementario.
RECOMENDACIONES:
- Se sugiere ecografía mamaria para post

In [11]:
# Exportar 3 ejemplos de recomendación por cada BI-RADS a un archivo TXT
with open("ejemplos_recomendaciones.txt", "w", encoding="utf-8") as f:
    f.write("EJEMPLOS DE RECOMENDACIONES REALES POR BI-RADS\n")
    f.write("=" * 80 + "\n")
    
    for birads in sorted(df["BI-RADS"].unique()):
        grupo = df[(df["BI-RADS"] == birads) & 
                   (df["Recommendations"] != "sin_recomendacion")]
        
        n_disponibles = len(grupo)
        n_mostrar = min(3, n_disponibles)
        
        f.write(f"\n\n{'#' * 80}\n")
        f.write(f"# BI-RADS {birads}  (mostrando {n_mostrar} de {n_disponibles} disponibles)\n")
        f.write(f"{'#' * 80}\n")
        
        if n_mostrar == 0:
            f.write("\n  (sin ejemplos con recomendación poblada)\n")
            continue
        
        muestra = grupo.sample(n_mostrar, random_state=42)
        for i, (idx, row) in enumerate(muestra.iterrows(), 1):
            f.write(f"\n[Ejemplo {i}] (índice {idx})\n")
            f.write(f"  Recommendations:\n")
            f.write(f"  {row['Recommendations']}\n")

print("Archivo guardado: ejemplos_recomendaciones.txt")
print("Ábrelo en VS Code (debería estar en la misma carpeta que el notebook)")

Archivo guardado: ejemplos_recomendaciones.txt
Ábrelo en VS Code (debería estar en la misma carpeta que el notebook)


Revision de los parametro de informacion previa en el infrome para sacar resomendacion mas especifica

In [12]:
import re

# Patrones de contexto clínico que pueden estar implícitos en el Full_Report
patrones_contexto = {
    "biopsia_previa": [
        r"biopsia\s+(core|previa|incisional|escisional|por\s+aguja)",
        r"antecedente.{0,30}biopsia",
        r"estudio\s+histol[oó]gico\s+previo",
        r"con\s+histolog[ií]a",
        r"histopatolog[ií]a\s+(previa|conocida|informada)",
    ],
    "cancer_previo_tratamiento": [
        r"operad[ao]\s+de\s+(c[aá]ncer|carcinoma)",
        r"post[\s-]?mastectom[ií]a",
        r"post[\s-]?quimioterapia",
        r"post[\s-]?radioterapia",
        r"seguimiento\s+oncol[oó]gico",
        r"antecedente.{0,30}(c[aá]ncer|carcinoma)",
        r"tumorectom[ií]a",
        r"cuadrantectom[ií]a",
    ],
    "control_evolutivo": [
        r"control\s+de\s+lesi[oó]n",
        r"control\s+evolutivo",
        r"sin\s+cambios\s+respecto",
        r"estable\s+respecto",
        r"comparad[ao]\s+con\s+estudio",
        r"comparativo\s+con",
        r"lesi[oó]n\s+(estable|conocida)",
    ],
}

# Marcar cada informe con los contextos detectados
for nombre, patrones in patrones_contexto.items():
    patron_compuesto = "|".join(patrones)
    df[f"ctx_{nombre}"] = df["Full_Report"].str.contains(
        patron_compuesto, case=False, regex=True, na=False
    )

# Distribución general
print("=" * 70)
print("CONTEXTO CLÍNICO IMPLÍCITO EN Full_Report")
print("=" * 70)
print(f"Total informes: {len(df)}\n")

for nombre in patrones_contexto:
    col = f"ctx_{nombre}"
    n = df[col].sum()
    pct = 100 * n / len(df)
    print(f"  {nombre:30s}: {n:4d} informes ({pct:.1f}%)")

# Cruce con BI-RADS: qué contextos aparecen más en cada categoría
print("\n" + "=" * 70)
print("CRUCE: contextos clínicos por categoría BI-RADS")
print("=" * 70)

for birads in sorted(df["BI-RADS"].unique()):
    grupo = df[df["BI-RADS"] == birads]
    n_total = len(grupo)
    print(f"\nBI-RADS {birads} (n={n_total}):")
    for nombre in patrones_contexto:
        col = f"ctx_{nombre}"
        n_con_ctx = grupo[col].sum()
        pct = 100 * n_con_ctx / n_total if n_total > 0 else 0
        print(f"    {nombre:30s}: {n_con_ctx:4d} ({pct:.1f}%)")

/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_9566/851205650.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[f"ctx_{nombre}"] = df["Full_Report"].str.contains(
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_9566/851205650.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[f"ctx_{nombre}"] = df["Full_Report"].str.contains(
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_9566/851205650.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[f"ctx_{nombre}"] = df["Full_Report"].str.contains(


CONTEXTO CLÍNICO IMPLÍCITO EN Full_Report
Total informes: 4357

  biopsia_previa                :    1 informes (0.0%)
  cancer_previo_tratamiento     :    0 informes (0.0%)
  control_evolutivo             :    1 informes (0.0%)

CRUCE: contextos clínicos por categoría BI-RADS

BI-RADS 0 (n=966):
    biopsia_previa                :    0 (0.0%)
    cancer_previo_tratamiento     :    0 (0.0%)
    control_evolutivo             :    0 (0.0%)

BI-RADS 1 (n=596):
    biopsia_previa                :    0 (0.0%)
    cancer_previo_tratamiento     :    0 (0.0%)
    control_evolutivo             :    1 (0.2%)

BI-RADS 2 (n=2635):
    biopsia_previa                :    1 (0.0%)
    cancer_previo_tratamiento     :    0 (0.0%)
    control_evolutivo             :    0 (0.0%)

BI-RADS 3 (n=87):
    biopsia_previa                :    0 (0.0%)
    cancer_previo_tratamiento     :    0 (0.0%)
    control_evolutivo             :    0 (0.0%)

BI-RADS 4 (n=52):
    biopsia_previa                :    0 (0.0%)

In [13]:
import re

# Comprobación 1: buscar la palabra "biopsia" en cualquier forma, sin patrón estricto
n_biopsia = df["Full_Report"].str.contains(r"biopsia", case=False, regex=True, na=False).sum()
n_histol = df["Full_Report"].str.contains(r"histol", case=False, regex=True, na=False).sum()
n_carcinoma = df["Full_Report"].str.contains(r"carcinoma|c[aá]ncer", case=False, regex=True, na=False).sum()
n_mastectomia = df["Full_Report"].str.contains(r"mastectom", case=False, regex=True, na=False).sum()
n_antecedente = df["Full_Report"].str.contains(r"antecedente", case=False, regex=True, na=False).sum()
n_previo = df["Full_Report"].str.contains(r"\bprevi[ao]", case=False, regex=True, na=False).sum()
n_seguimiento = df["Full_Report"].str.contains(r"seguimiento", case=False, regex=True, na=False).sum()
n_comparativo = df["Full_Report"].str.contains(r"comparativ|compara con", case=False, regex=True, na=False).sum()

print("=" * 60)
print("MENCIONES DE PALABRAS CLAVE EN Full_Report")
print("=" * 60)
print(f"  biopsia       : {n_biopsia:4d} ({100*n_biopsia/len(df):.1f}%)")
print(f"  histol*       : {n_histol:4d} ({100*n_histol/len(df):.1f}%)")
print(f"  carcinoma/cáncer: {n_carcinoma:4d} ({100*n_carcinoma/len(df):.1f}%)")
print(f"  mastectom*    : {n_mastectomia:4d} ({100*n_mastectomia/len(df):.1f}%)")
print(f"  antecedente   : {n_antecedente:4d} ({100*n_antecedente/len(df):.1f}%)")
print(f"  previo/previa : {n_previo:4d} ({100*n_previo/len(df):.1f}%)")
print(f"  seguimiento   : {n_seguimiento:4d} ({100*n_seguimiento/len(df):.1f}%)")
print(f"  comparativ*   : {n_comparativo:4d} ({100*n_comparativo/len(df):.1f}%)")

# Comprobación 2: ver el Full_Report completo de los pocos casos BI-RADS 4 para entender qué dicen
print("\n" + "=" * 60)
print("LOS 5 PRIMEROS INFORMES BI-RADS 4 (Full_Report completo)")
print("=" * 60)

birads4 = df[df["BI-RADS"] == 4].head(5)
for i, (idx, row) in enumerate(birads4.iterrows(), 1):
    print(f"\n----- Informe {i} (idx {idx}) -----")
    print(row["Full_Report"])
    print()

MENCIONES DE PALABRAS CLAVE EN Full_Report
  biopsia       :   36 (0.8%)
  histol*       :   59 (1.4%)
  carcinoma/cáncer:    5 (0.1%)
  mastectom*    :    0 (0.0%)
  antecedente   :  181 (4.2%)
  previo/previa :  904 (20.7%)
  seguimiento   :    0 (0.0%)
  comparativ*   :  918 (21.1%)

LOS 5 PRIMEROS INFORMES BI-RADS 4 (Full_Report completo)

----- Informe 1 (idx 112) -----
MAMOGRAFÍA DIGITAL CRÁNEO-CAUDAL Y MEDIO LATERAL OBLÍCUA DE AMBAS MAMAS Y LATERAL ESTRICTA DE LA MAMA IZQUIERDA
NO SE OBSERVAN ALTERACIONES CUTÁNEAS NI DEL MAMELÓN.
LAS MAMAS ESTÁN COMPUESTAS POR TEJIDO ADIPOSO CASI EN SU TOTALIDAD, OBSERVÁNDOSE UN REMANENTE DE TEJIDO GLANDULAR EN AMBAS REGIONES RETROAREOLARES (MAMAS TIPO A).
EN EL CSI, PLANO PROFUNDO DE LA MAMA IZQUIERDA, SE VISUALIZA PARCIALMENTE UNA IMAGEN NODULAR OVALADA, HIPERDENSA, CIRCUSNCRITA, DE APROXIMADAMENTE 30 MM DE DIÁMETRO MAYOR, LA CUAL REQUERIRÁ DE EVALUACIÓN COMPLEMENTARIA.
NO SE OBSERVAN CALCIFICACIONES SOSPECHOSAS.
SE EVIDENCIAN CALCIFICACIONES 

In [14]:
import re
from collections import Counter

# Patrón muy permisivo: captura cualquier cosa que parezca BI-RADS
# y el contexto inmediato (lo que viene después)
patron_birads = re.compile(
    r"(bi[-\s]?rad[s]?[\s®®]*[:\s]*\(?[\s]*\d[abc]?)",
    re.IGNORECASE
)

# Recolectar todas las menciones del corpus
todas_menciones = []
for texto in df["Full_Report"]:
    if not isinstance(texto, str):
        continue
    matches = patron_birads.findall(texto)
    for m in matches:
        # Normalizar para conteo
        normalizado = re.sub(r"\s+", " ", m.strip().lower())
        todas_menciones.append(normalizado)

# Top 30 variantes más frecuentes
contador = Counter(todas_menciones)
print("=" * 70)
print("TOP 30 VARIANTES DE 'BI-RADS' EN EL CORPUS")
print("=" * 70)
print(f"Total menciones encontradas: {len(todas_menciones)}")
print(f"Variantes únicas: {len(contador)}\n")

for variante, n in contador.most_common(30):
    print(f"  {n:5d} × '{variante}'")

# Análisis adicional: cuántos informes tienen al menos UNA mención
n_con_mencion = 0
n_sin_mencion = 0
for texto in df["Full_Report"]:
    if isinstance(texto, str) and patron_birads.search(texto):
        n_con_mencion += 1
    else:
        n_sin_mencion += 1

print(f"\n" + "=" * 70)
print("COBERTURA DEL PATRÓN")
print("=" * 70)
print(f"Informes con AL MENOS UNA mención de BI-RADS: {n_con_mencion} ({100*n_con_mencion/len(df):.1f}%)")
print(f"Informes SIN ninguna mención visible:         {n_sin_mencion} ({100*n_sin_mencion/len(df):.1f}%)")

# Ver los informes sin mención (si los hay)
if n_sin_mencion > 0:
    print(f"\n--- 3 ejemplos de informes SIN mención BI-RADS ---")
    sin_mencion_df = df[~df["Full_Report"].str.contains(patron_birads, regex=True, na=False)]
    for i, (idx, row) in enumerate(sin_mencion_df.head(3).iterrows(), 1):
        print(f"\n[Ejemplo {i}] idx {idx} | BI-RADS etiqueta={row['BI-RADS']}")
        print(f"Full_Report:\n{row['Full_Report']}")
        print("---")

TOP 30 VARIANTES DE 'BI-RADS' EN EL CORPUS
Total menciones encontradas: 4357
Variantes únicas: 26

   1575 × 'bi-rads 2'
   1060 × 'bi-rads ® 2'
    529 × 'bi-rads 0'
    436 × 'bi-rads ® 0'
    304 × 'bi-rads 1'
    288 × 'bi-rads ® 1'
     54 × 'bi-rads ® 3'
     32 × 'bi-rads 3'
     11 × 'bi-rads 4'
      9 × 'bi-rads 4c'
      8 × 'bi-rads 5'
      8 × 'bi-rads ® 5'
      8 × 'bi-rads ® 4'
      7 × 'bi-rads 4a'
      6 × 'bi-rads ® 4a'
      5 × 'bi-rads 4b'
      3 × 'bi-rads ® 4c'
      3 × 'bi-rads ® 6'
      2 × 'bi-rads0'
      2 × 'bi-rads 6'
      2 × 'bi-rads® 0'
      1 × 'bi-rads ®1'
      1 × 'bi-rads ® 4b'
      1 × 'bi-rads® 3'
      1 × 'bi-rads® 2'
      1 × 'bi-rads® 4a'

COBERTURA DEL PATRÓN
Informes con AL MENOS UNA mención de BI-RADS: 4355 (100.0%)
Informes SIN ninguna mención visible:         2 (0.0%)

--- 3 ejemplos de informes SIN mención BI-RADS ---

[Ejemplo 1] idx 43 | BI-RADS etiqueta=0
Full_Report:
MAMOGRAFÍA DIGITAL BILATERAL CRÁNEO-CAUDAL Y MEDIO LATE

/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_9566/394079932.py:51: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sin_mencion_df = df[~df["Full_Report"].str.contains(patron_birads, regex=True, na=False)]


In [15]:
# ¿Cuántas menciones de BI-RADS por informe?
def contar_menciones_birads(texto):
    if not isinstance(texto, str):
        return 0
    return len(patron_birads.findall(texto))

df["n_menciones_birads"] = df["Full_Report"].apply(contar_menciones_birads)

print("Distribución del número de menciones BI-RADS por informe:")
print(df["n_menciones_birads"].value_counts().sort_index())

# Ver un par de ejemplos con MÚLTIPLES menciones
print("\n--- Ejemplos con 2 menciones BI-RADS ---")
con_2 = df[df["n_menciones_birads"] == 2].head(2)
for idx, row in con_2.iterrows():
    print(f"\nidx {idx} (etiqueta {row['BI-RADS']}):")
    matches = patron_birads.findall(row["Full_Report"])
    for m in matches:
        print(f"  → {m}")

# Si hay con 3+
con_3_mas = df[df["n_menciones_birads"] >= 3]
if len(con_3_mas) > 0:
    print(f"\n--- {len(con_3_mas)} informes con 3+ menciones, mostrando 1 ---")
    idx = con_3_mas.iloc[0].name
    row = con_3_mas.iloc[0]
    print(f"\nidx {idx} (etiqueta {row['BI-RADS']}):")
    matches = patron_birads.findall(row["Full_Report"])
    for m in matches:
        print(f"  → {m}")

Distribución del número de menciones BI-RADS por informe:
n_menciones_birads
0       2
1    4353
2       2
Name: count, dtype: int64

--- Ejemplos con 2 menciones BI-RADS ---

idx 1079 (etiqueta 1):
  → BI-RADS 1
  → BI-RADS 0

idx 1111 (etiqueta 1):
  → BI-RADS 1
  → BI-RADS 2


In [16]:
import re

# Re-buscar con el patrón pero con más contexto
patron = re.compile(r"bi[-\s]?rad[s]?[\s®®]*[:\s]*\(?[\s]*\d[abc]?", re.IGNORECASE)

# Ver los dos informes con 2 menciones en detalle
con_2 = df[df["n_menciones_birads"] == 2]

for idx, row in con_2.iterrows():
    texto = row["Full_Report"]
    print(f"\n{'=' * 70}")
    print(f"idx {idx} (etiqueta REAL = BI-RADS {row['BI-RADS']})")
    print(f"{'=' * 70}")
    
    # Encontrar todas las menciones con sus posiciones
    for i, m in enumerate(patron.finditer(texto), 1):
        inicio = max(0, m.start() - 60)
        fin = min(len(texto), m.end() + 60)
        snippet = texto[inicio:fin].replace("\n", " ").replace("  ", " ")
        print(f"\nMención {i}: ...{snippet}...")


idx 1079 (etiqueta REAL = BI-RADS 1)

Mención 1: ...ORMA ESTUDIO SIN EVIDENCIA DE IMÁGENES SÓLIDAS NI LÍQUIDAS (BI-RADS 1). CONCLUSIÓN: - IMÁGENES NODULARES AMBAS MAMAS, LA DEL LADO...

Mención 2: ...CALCIFICACIONES CON CARACTERES BENIGNOS EN AMBAS MAMAS. -  BI-RADS 0 (Según la ACR). Requerirá de estudio complementario. RECOME...

idx 1111 (etiqueta REAL = BI-RADS 1)

Mención 1: ... 13-09-18. ECOGRAFÍA MAMARIA PRACTICADA EL 20-09-18 INFORMA BI-RADS 1. CONCLUSIÓN: - CALCIFICACIONES CON CARACTERES BENIGNOS EN A...

Mención 2: ...- CALCIFICACIONES CON CARACTERES BENIGNOS EN AMBAS MAMAS. - BI-RADS 2 (Según la ACR). RECOMENDACIONES: - SE SUGIERE CONTROL MAMOG...


In [17]:
import re

# Variantes posibles de la palabra "conclusión"
patrones_conclusion = {
    "CONCLUSIÓN: (con dos puntos)": r"conclusi[oó]n\s*:",
    "VALORACIÓN: (variante)": r"valoraci[oó]n\s*:",
    "IMPRESIÓN: (variante)": r"impresi[oó]n\s*:",
    "DIAGNÓSTICO: (variante)": r"diagn[oó]stico\s*:",
}

print("=" * 70)
print("FRECUENCIA DE ENCABEZADOS DE CONCLUSIÓN EN EL CORPUS")
print("=" * 70)

for nombre, patron in patrones_conclusion.items():
    n = df["Full_Report"].str.contains(patron, case=False, regex=True, na=False).sum()
    pct = 100 * n / len(df)
    print(f"  {nombre:35s}: {n:4d} ({pct:.1f}%)")

# ¿Cuántos informes tienen AL MENOS un encabezado de conclusión?
patron_cualquier = "|".join(patrones_conclusion.values())
df["tiene_conclusion"] = df["Full_Report"].str.contains(
    patron_cualquier, case=False, regex=True, na=False
)
n_con = df["tiene_conclusion"].sum()
n_sin = len(df) - n_con
print(f"\n  Con algún encabezado de conclusión: {n_con} ({100*n_con/len(df):.1f}%)")
print(f"  Sin ningún encabezado de conclusión:  {n_sin} ({100*n_sin/len(df):.1f}%)")

# Ver 2 ejemplos sin encabezado de conclusión (si los hay)
if n_sin > 0 and n_sin <= 10:
    print("\n--- Los informes SIN encabezado de conclusión ---")
    for idx, row in df[~df["tiene_conclusion"]].head(3).iterrows():
        print(f"\nidx {idx} | BI-RADS {row['BI-RADS']}:")
        print(row["Full_Report"][:400])
elif n_sin > 10:
    print(f"\n(hay {n_sin} sin encabezado, demasiados para mostrar todos)")

FRECUENCIA DE ENCABEZADOS DE CONCLUSIÓN EN EL CORPUS
  CONCLUSIÓN: (con dos puntos)       : 2949 (67.7%)
  VALORACIÓN: (variante)             : 1408 (32.3%)
  IMPRESIÓN: (variante)              :    0 (0.0%)
  DIAGNÓSTICO: (variante)            :    0 (0.0%)

  Con algún encabezado de conclusión: 4357 (100.0%)
  Sin ningún encabezado de conclusión:  0 (0.0%)


In [18]:
import re

# Buscar BI-RADS seguido de número romano (con variantes de escritura)
patron_romano = re.compile(
    r"bi[-\s*.]?rad[s]?[\s®]*[:\s]*\(?\s*(VI|IV|V|III|II|I)\b",
    re.IGNORECASE
)

# Buscar BI-RADS seguido de número escrito en palabras
patron_letras = re.compile(
    r"bi[-\s*.]?rad[s]?[\s®]*[:\s]*\(?\s*(cero|uno|dos|tres|cuatro|cinco|seis)\b",
    re.IGNORECASE
)

# Aplicar
df["tiene_romano"] = df["Full_Report"].str.contains(patron_romano, regex=True, na=False)
df["tiene_letras"] = df["Full_Report"].str.contains(patron_letras, regex=True, na=False)

n_romano = df["tiene_romano"].sum()
n_letras = df["tiene_letras"].sum()

print("=" * 70)
print("BÚSQUEDA DE BI-RADS CON OTRAS NUMERACIONES")
print("=" * 70)
print(f"Con número romano (I, II, III, IV, V, VI): {n_romano} informes ({100*n_romano/len(df):.2f}%)")
print(f"Con número en palabras (uno, dos, tres...):  {n_letras} informes ({100*n_letras/len(df):.2f}%)")

# Ver ejemplos si existen
if n_romano > 0:
    print("\n--- 3 ejemplos con número romano ---")
    for idx, row in df[df["tiene_romano"]].head(3).iterrows():
        match = patron_romano.search(row["Full_Report"])
        if match:
            inicio = max(0, match.start() - 40)
            fin = min(len(row["Full_Report"]), match.end() + 60)
            snippet = row["Full_Report"][inicio:fin].replace("\n", " ")
            print(f"\nidx {idx} | BI-RADS etiqueta={row['BI-RADS']}:")
            print(f"  ...{snippet}...")

if n_letras > 0:
    print("\n--- 3 ejemplos con número en palabras ---")
    for idx, row in df[df["tiene_letras"]].head(3).iterrows():
        match = patron_letras.search(row["Full_Report"])
        if match:
            inicio = max(0, match.start() - 40)
            fin = min(len(row["Full_Report"]), match.end() + 60)
            snippet = row["Full_Report"][inicio:fin].replace("\n", " ")
            print(f"\nidx {idx} | BI-RADS etiqueta={row['BI-RADS']}:")
            print(f"  ...{snippet}...")

if n_romano == 0 and n_letras == 0:
    print("\n✓ Este corpus solo usa números arábigos para BI-RADS.")

BÚSQUEDA DE BI-RADS CON OTRAS NUMERACIONES
Con número romano (I, II, III, IV, V, VI): 0 informes (0.00%)
Con número en palabras (uno, dos, tres...):  0 informes (0.00%)

✓ Este corpus solo usa números arábigos para BI-RADS.


/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_9566/2273646360.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["tiene_romano"] = df["Full_Report"].str.contains(patron_romano, regex=True, na=False)
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_9566/2273646360.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["tiene_letras"] = df["Full_Report"].str.contains(patron_letras, regex=True, na=False)
